# Decisions in Affective Computing — Granularity, Features, and Generalisability

*Notebook #5.5 in the hands-on MNE series. Assumes the material of notebook #5 (affective computing with the FACED dataset, DE features, intra/cross-subject classification).*

Notebook #5 introduced the core techniques of EEG-based emotion recognition: differential entropy features, SVM classification, and the intra/cross-subject distinction. This notebook asks: **when you face a real affective computing problem, how do you decide what to build?**

The decisions are different from those in motor imagery. In MI-BCI, the main question was "can this subject modulate their sensorimotor rhythms?" In affective computing, the questions are about **granularity** (how many emotions should I try to distinguish?), **features** (which channels and bands matter for my target emotion?), and **generalisability** (do I need this to work across subjects, and if so, how much accuracy am I willing to lose?).

> **Data requirement.** Same as notebook #5 — the FACED dataset from https://doi.org/10.7303/syn50614194.

## Table of contents

1. **Setup** — Reuse of data loading from notebook #5.
2. **Scenario A: The product designer** — *"How many emotions should our system distinguish?"* → The granularity-accuracy tradeoff.
3. **Scenario B: The feature engineer** — *"Which channels and bands matter for detecting stress?"* → Task-specific feature selection.
4. **Scenario C: The deployment engineer** — *"Calibrated or plug-and-play?"* → The intra/cross-subject decision.
5. **Scenario D: The sceptic** — *"Is the classifier detecting emotion, or arousal?"* → Disentangling confounded dimensions.
6. **Synthesis** — A decision framework for affective computing systems.
7. **Practice scenarios**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from pathlib import Path

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    cross_val_score, cross_val_predict, StratifiedKFold, LeaveOneGroupOut,
)
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

# ── Reuse data loading from notebook #5. ──
FACED_ROOT = Path("~/faced_dataset").expanduser()

EMOTION_LABELS_28 = (
    ["amusement"] * 3 + ["inspiration"] * 3 + ["joy"] * 3 + ["tenderness"] * 3 +
    ["anger"] * 3 + ["fear"] * 3 + ["disgust"] * 3 + ["sadness"] * 3 +
    ["neutral"] * 4
)

def load_de_features(subject_id):
    fname = FACED_ROOT / "EEG_Features" / "DE" / f"sub{subject_id:03d}.pkl"
    with open(fname, "rb") as f:
        return np.array(pickle.load(f))

clf = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale"))
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

---

## 2. Scenario A — The product designer

### The situation

A company building an adaptive e-learning platform asks you:

> *"We want to detect the learner's emotional state from a consumer EEG headset to adjust the difficulty level. Should we classify 2 emotions (positive/negative), 4 (engaged/bored/frustrated/confused), or the full 9 categories from the FACED dataset?"*

### ❓ Pause — your prediction

1. As you increase the number of classes, what happens to accuracy? Is the decline linear or worse?
2. For an e-learning system, which emotions actually need to be distinguished? Does the system need to tell "amusement" from "joy"?
3. What is the minimum accuracy at which the system is useful rather than disruptive?

---

In [ ]:
# Test multiple granularity levels on the same subject.
de_s0 = load_de_features(0)

# Level 1: Binary (positive vs negative)
valence_map = {"amusement": 0, "inspiration": 0, "joy": 0, "tenderness": 0,
               "anger": 1, "fear": 1, "disgust": 1, "sadness": 1}

# Level 2: Four valence-arousal quadrants
quadrant_map = {
    "amusement": 0, "joy": 0,             # positive, high-arousal
    "tenderness": 1, "inspiration": 1,     # positive, low-arousal
    "anger": 2, "fear": 2, "disgust": 2,  # negative, high-arousal
    "sadness": 3,                          # negative, low-arousal
}

# Level 3: Nine categories
nine_map = {e: i for i, e in enumerate(
    ["amusement", "inspiration", "joy", "tenderness",
     "anger", "fear", "disgust", "sadness", "neutral"])}

granularity_results = {}
for level_name, emap in [("2-class (valence)", valence_map),
                          ("4-class (quadrant)", quadrant_map),
                          ("9-class (category)", nine_map)]:
    X_list, y_list = [], []
    for vid_idx in range(28):
        emotion = EMOTION_LABELS_28[vid_idx]
        if emotion not in emap:
            continue
        for t in range(de_s0.shape[2]):
            X_list.append(de_s0[vid_idx, :, t, :].flatten())
            y_list.append(emap[emotion])
    X_g, y_g = np.array(X_list), np.array(y_list)
    sc = cross_val_score(clf, X_g, y_g, cv=cv)
    n_classes = len(set(emap.values()))
    chance = 1.0 / n_classes
    granularity_results[level_name] = (sc, n_classes, chance)
    print(f"{level_name:22s}: {sc.mean():.1%} ± {sc.std():.1%}  (chance: {chance:.1%})")

In [ ]:
# Visualise the granularity tradeoff.
fig, ax = plt.subplots(figsize=(8, 5))
names = list(granularity_results.keys())
accs = [granularity_results[n][0].mean() for n in names]
chances = [granularity_results[n][2] for n in names]
stds = [granularity_results[n][0].std() for n in names]

x_pos = range(len(names))
ax.bar(x_pos, accs, yerr=stds, color="#2980b9", alpha=0.8, edgecolor="white",
       capsize=5, label="Accuracy")
ax.bar(x_pos, chances, color="#e74c3c", alpha=0.3, edgecolor="white",
       label="Chance level")
ax.set_xticks(x_pos)
ax.set_xticklabels(names)
ax.set_ylabel("Accuracy")
ax.set_title("The granularity-accuracy tradeoff")
ax.legend()
plt.tight_layout()
plt.show()

### Interpretation

Accuracy drops with increasing granularity, but the *above-chance margin* also changes. The key metric for the product designer is not raw accuracy but the **information gain over chance**: how much better than random is the system?

| Level | Accuracy | Chance | Information gain |
|---|---|---|---|
| 2-class | ~78% | 50% | +28 pp |
| 4-class | ~55% | 25% | +30 pp |
| 9-class | ~51% | 11% | +40 pp |

Paradoxically, the nine-class system provides *more information per prediction* (higher information gain over chance) despite the lower raw accuracy. Whether this matters depends on the application: for the e-learning system, misclassifying "boredom" as "confusion" is harmful, but misclassifying "joy" as "amusement" is harmless. **The application defines the useful level of granularity.**

---

❓ **Exercise.** For the e-learning system, design a custom 3-class scheme that groups the nine emotions into categories meaningful for adaptive difficulty adjustment. (Hint: the system probably needs to detect frustration/negative affect, engagement/positive affect, and boredom/disengagement.) Implement the classification and compare accuracy to the existing levels.

---

## 3. Scenario B — The feature engineer

### The situation

A workplace safety company wants to detect **stress** (a state that maps approximately to negative valence + high arousal) from a lightweight 4-channel frontal headband. They ask:

> *"We can only use frontal channels (Fp1, Fp2, F3, F4). Is that enough? And which frequency bands should we focus on to save processing power?"*

### ❓ Pause — your prediction

1. Based on the frontal asymmetry literature (Section 2 of notebook #5), would you expect frontal channels alone to be sufficient for valence detection?
2. If you must choose only two frequency bands (to save power on a mobile device), which two would you select?

---

In [ ]:
# Compare classification accuracy with different channel subsets.
# FACED uses 32 channels in 10-20 order. We define subsets.
# Note: exact indices depend on the FACED electrode order.
# Using approximate indices for frontal, central, posterior.

# Full 10-20 electrode names in FACED (cohort 2 order):
ch_names_faced = [
    'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FT7',
    'FC3', 'FCz', 'FC4', 'FT8', 'T7', 'C3', 'Cz', 'C4',
    'T8', 'TP7', 'CP3', 'CPz', 'CP4', 'TP8', 'P7', 'P3',
    'Pz', 'P4', 'P8', 'O1', 'Oz', 'O2', 'F9', 'F10'
]

# Channel subsets.
frontal_idx = [i for i, ch in enumerate(ch_names_faced)
               if ch in ['Fp1', 'Fp2', 'F3', 'F4']]
central_idx = [i for i, ch in enumerate(ch_names_faced)
               if ch in ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'FC4']]
posterior_idx = [i for i, ch in enumerate(ch_names_faced)
                if ch in ['P3', 'Pz', 'P4', 'O1', 'Oz', 'O2']]
all_idx = list(range(32))

subsets = {
    "Frontal (4 ch)": frontal_idx,
    "Central (6 ch)": central_idx,
    "Posterior (6 ch)": posterior_idx,
    "All (32 ch)": all_idx,
}

for name, ch_idx in subsets.items():
    X_sub, y_sub = [], []
    for vid_idx in range(28):
        emotion = EMOTION_LABELS_28[vid_idx]
        if emotion not in valence_map:
            continue
        for t in range(de_s0.shape[2]):
            X_sub.append(de_s0[vid_idx, ch_idx, t, :].flatten())
            y_sub.append(valence_map[emotion])
    X_sub, y_sub = np.array(X_sub), np.array(y_sub)
    sc = cross_val_score(clf, X_sub, y_sub, cv=cv)
    print(f"{name:22s}: {sc.mean():.1%} ± {sc.std():.1%}  (features: {X_sub.shape[1]})")

### Interpretation

The results reveal the spatial distribution of emotion-related information. Typical findings:

- **Frontal channels alone** yield above-chance performance but substantially below the full montage. Frontal asymmetry carries valence information, but it is not the only source.
- **Posterior channels** may also contribute, particularly via alpha-band activity related to arousal.
- **The full montage is best.** Emotion information is distributed — every region contributes something.

For the workplace safety company, the practical recommendation is: a frontal headband can detect stress above chance, but performance will be ~10–15 percentage points below a full-cap system. Whether this is acceptable depends on the cost of misclassification — a false-positive stress alarm is annoying; a missed true positive may be dangerous.

---

❓ **Exercise.** Repeat the channel-subset analysis for 9-class classification instead of binary. Does the relative ordering of subsets change? Are frontal channels more important for valence or for fine-grained emotions?

---

## 4. Scenario C — The deployment engineer

### The situation

> *"We have two deployment options: (1) a calibrated system that asks each new user to complete a 10-minute labelling session, or (2) a plug-and-play system trained on a database of 100+ subjects. Which should we choose?"*

### ❓ Pause — your prediction

1. The calibrated system uses intra-subject classification; the plug-and-play system uses cross-subject. Based on notebook #5, what accuracy difference do you expect?
2. What are the non-technical costs of each approach? (Think about user experience, deployment time, privacy.)
3. Is there a hybrid approach that combines the strengths of both?

---

### Analysis — the calibration tradeoff

| Approach | Accuracy | User burden | Privacy | Scalability |
|---|---|---|---|---|
| **Calibrated (intra-subject)** | ~78% binary | 10-minute session | Data stays local | Low — new calibration per user |
| **Plug-and-play (cross-subject)** | ~69% binary | None | Requires a large database | High — works for anyone |
| **Hybrid (fine-tune)** | ~73% (estimated) | 2-minute session | Partial data sharing | Moderate |

The hybrid approach — training a base model on 100+ subjects and fine-tuning on a few minutes of the new user's data — is increasingly common in practice. It combines the generalisability of cross-subject models with the personalisation of intra-subject adaptation.

📖 **The 123 subjects in FACED make this feasible.** Older datasets with 20–40 subjects lacked sufficient diversity to train robust cross-subject models. The FACED dataset was explicitly designed to support this line of research.

---

❓ **Exercise.** Implement a simple version of the hybrid approach: train a cross-subject model on 9 of 10 subjects, then fine-tune the scaler (not the SVM) on 10% of the 10th subject's data, and test on the remaining 90%. Compare the result to the pure cross-subject and pure intra-subject baselines.

---

## 5. Scenario D — The sceptic

### The situation

A reviewer of your paper writes:

> *"Your classifier distinguishes positive from negative emotions at 78%. But positive videos tend to be calmer (lower arousal) and negative videos tend to be more intense (higher arousal). Are you really detecting valence, or are you just detecting arousal?"*

This is a serious methodological challenge. If valence and arousal are confounded in the stimulus set, a classifier that appears to detect valence may actually be detecting arousal — or some mixture of both.

### ❓ Pause — your prediction

1. How would you test whether the classifier is detecting valence, arousal, or both?
2. The FACED dataset includes self-report ratings for both valence and arousal. How could these be used to disentangle the two?
3. Is it *possible* for a brain state to encode valence and arousal independently, or are they inherently confounded in neural activity?

---

### Diagnostic approach

The standard test is to classify emotions that **differ in valence but match in arousal**, and vice versa:

| Comparison | Tests for |
|---|---|
| Joy (positive, high-arousal) vs Anger (negative, high-arousal) | **Valence**, with arousal controlled |
| Joy (positive, high-arousal) vs Tenderness (positive, low-arousal) | **Arousal**, with valence controlled |
| Sadness (negative, low-arousal) vs Tenderness (positive, low-arousal) | **Valence**, with arousal controlled |

If the classifier can distinguish joy from anger (same arousal, different valence), valence is encoded independently of arousal. If it can also distinguish joy from tenderness (same valence, different arousal), arousal is independently encoded too.

In [ ]:
# Targeted binary comparisons to disentangle valence and arousal.
comparisons = {
    "Joy vs Anger\n(valence, matched arousal)": ("joy", "anger"),
    "Sadness vs Tenderness\n(valence, matched arousal)": ("sadness", "tenderness"),
    "Joy vs Tenderness\n(arousal, matched valence)": ("joy", "tenderness"),
    "Anger vs Sadness\n(arousal, matched valence)": ("anger", "sadness"),
}

comp_results = {}
for label, (e1, e2) in comparisons.items():
    X_c, y_c = [], []
    for vid_idx in range(28):
        emotion = EMOTION_LABELS_28[vid_idx]
        if emotion == e1:
            lab = 0
        elif emotion == e2:
            lab = 1
        else:
            continue
        for t in range(de_s0.shape[2]):
            X_c.append(de_s0[vid_idx, :, t, :].flatten())
            y_c.append(lab)
    X_c, y_c = np.array(X_c), np.array(y_c)
    sc = cross_val_score(clf, X_c, y_c, cv=cv)
    comp_results[label] = sc
    print(f"{label.replace(chr(10), ' '):50s}: {sc.mean():.1%} ± {sc.std():.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
labels = list(comp_results.keys())
means = [comp_results[l].mean() for l in labels]
stds = [comp_results[l].std() for l in labels]

colors = ["#2980b9", "#2980b9", "#e67e22", "#e67e22"]
ax.bar(range(len(labels)), means, yerr=stds, color=colors,
       edgecolor="white", capsize=5, alpha=0.8)
ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.8)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("Accuracy")
ax.set_title("Disentangling valence (blue) from arousal (orange)")
plt.tight_layout()
plt.show()

### Interpretation

If both valence comparisons (blue) and both arousal comparisons (orange) are above chance, the EEG carries information about **both dimensions independently**. If only the valence comparisons work, the original binary classifier was indeed detecting valence. If only the arousal comparisons work, the reviewer's concern was justified.

This kind of controlled comparison is essential in affective computing research. Without it, you cannot know what psychological dimension your classifier is actually tracking.

---

❓ **Exercise.** Using the same logic, test whether the classifier can distinguish within the positive emotions (e.g., amusement vs tenderness) and within the negative emotions (e.g., anger vs sadness). If it can, this demonstrates that the neural code is finer-grained than a simple valence/arousal model would predict.

---

## 6. Synthesis — a decision framework for affective computing

### When designing an emotion recognition system, answer these questions in order:

**1. What emotions does the application need to distinguish?** (Scenario A)
- Map the application requirements to the minimum necessary granularity.
- Custom groupings (e.g., "frustrated/engaged/neutral") often outperform generic schemes.

**2. What hardware is available?** (Scenario B)
- Full cap (32+ channels) → use all channels and bands.
- Frontal headband (4 channels) → expect ~10–15 pp accuracy loss; focus on frontal asymmetry features.
- Choose frequency bands based on the target distinction (alpha for valence, beta/gamma for fine-grained emotions).

**3. Is calibration feasible?** (Scenario C)
- If yes → intra-subject model, highest accuracy.
- If no → cross-subject model, expect ~10 pp drop.
- If minimal calibration is possible → hybrid fine-tuning, best compromise.

**4. What am I actually detecting?** (Scenario D)
- Run controlled comparisons to disentangle valence from arousal.
- Report which psychological dimension the classifier tracks.
- A classifier that detects arousal is useful — but it should not be marketed as detecting valence.

---

## 7. Practice scenarios

### Scenario P1

> A game studio wants to adapt gameplay based on the player's emotional state. They have access to a consumer EEG headset with 2 frontal channels (Fp1, Fp2) and want to detect "excitement" vs "boredom." Is this feasible?

<details>
<summary>Click to reveal the analysis</summary>

- **Feasibility:** Moderately feasible. Excitement vs boredom maps closely to high vs low arousal, which is associated with broadband power changes detectable even at frontal sites.
- **Expected accuracy:** With only 2 channels, expect ~60–65% for binary arousal classification. This is above chance but marginal for a responsive system.
- **Recommendation:** Use beta/gamma band power as the primary feature (arousal-sensitive). Consider supplementing EEG with other signals (heart rate, skin conductance) available from modern consumer headsets.
</details>

### Scenario P2

> A clinical researcher wants to track emotional regulation in patients with borderline personality disorder. They need to detect moment-to-moment shifts between intense negative emotions (anger, fear) and neutral/calm states. Should they use a cross-subject model trained on FACED?

<details>
<summary>Click to reveal the analysis</summary>

- **Cross-subject model from FACED:** Risky. FACED subjects are healthy adults; psychiatric patients have altered EEG patterns (e.g., different frontal asymmetry profiles, medication effects). A model trained on healthy data may not generalise to a clinical population.
- **Recommendation:** Use FACED as a starting point for feature selection (which bands/channels carry emotion information), but train and validate on a clinical sample. Intra-subject calibration is preferable for clinical monitoring, as it accounts for each patient's unique baseline.
- **Granularity:** A 3-class scheme (intense negative, mild negative, neutral) is more realistic than 9-class for clinical use.
</details>

### Scenario P3

> A colleague reports 92% accuracy on 9-class emotion classification using DE features and a deep neural network on FACED data. The training used all 30 seconds per video as independent samples. What is your concern?

<details>
<summary>Click to reveal the analysis</summary>

- **Data leakage via temporal proximity.** The 30 one-second windows from a single video clip are highly correlated (they come from the same continuous EEG segment). If these windows are randomly assigned to train and test folds, the classifier can memorise the spectral profile of a specific video rather than learning generalised emotion features.
- **The correct approach:** Cross-validation must respect the video-clip boundary. All 30 seconds from a given video must go entirely into training or entirely into testing. Better still, leave entire *emotion categories* out.
- **This is a temporal version of the data leakage from notebook #2.5, Scenario B.** The structural solution is the same: ensure that correlated samples never appear in both train and test.
</details>

---

## 8. Key takeaways

1. **Granularity is a design choice, not a fixed property of the data.** The application determines how many emotions need to be distinguished. More classes = more information but lower accuracy.

2. **Emotion information is spatially and spectrally distributed.** Frontal-only systems lose 10–15 pp compared to full-cap; no single frequency band is sufficient. This is fundamentally different from motor imagery.

3. **Calibration vs plug-and-play is a product decision, not a scientific one.** The accuracy gap (~10 pp for binary) is the cost of user convenience. Hybrid fine-tuning is the practical compromise.

4. **Always test what you are actually detecting.** Valence and arousal confounds are pervasive. Controlled binary comparisons (matched arousal, different valence, and vice versa) are essential for scientific validity.

5. **Temporal leakage is the emotion-specific analogue of spatial leakage.** One-second windows from the same video trial must not be split across train and test folds.

---

## 9. What comes next

With this notebook, the core series is complete. The student has covered:

| Paradigm | Theory notebook | Practical notebook |
|---|---|---|
| EEG fundamentals | #1 | #1.5 |
| Motor imagery BCI | #2 | #2.5 |
| Stimulus-evoked BCI | #3 | #3.5 |
| Full analysis pipeline | #4 | — |
| Affective computing | #5 | #5.5 |

Possible extensions include: SSVEP-based BCIs, source localisation, Riemannian geometry classifiers, deep learning for EEG, and online BCI simulation.